In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import geemap

from src.ndvi_ndwi import add_indices
from src.preprocess.aoi import load_city_aoi
from src.preprocess.gee_auth import initialize_ee
from src.preprocess.get_city_boundaries import Cities
from src.preprocess.load_and_preprocess_image import get_one_year_composite

initialize_ee()

In [6]:
CITY = Cities.BERLIN.value
YEAR = 2025

aoi = load_city_aoi(CITY.name)
image = add_indices(get_one_year_composite(aoi, YEAR))

In [7]:
ndvi_vis = {"min": -1, "max": 1, "palette": ["#d73027", "#ffffbf", "#1a9850"]}

Map = geemap.Map()
Map.centerObject(aoi, zoom=11)
Map.addLayer(image.select("NDVI"), ndvi_vis, "NDVI")
Map.add_colorbar(ndvi_vis, label="NDVI")
Map

Map(center=[52.50148692830281, 13.402429744019866], controls=(WidgetControl(options=['position', 'transparent_…

In [8]:
import ee

from src.preprocess.load_and_preprocess_image import (
    END_YEAR,
    START_YEAR,
    get_one_year_composite,
)


def compute_ndvi_trend(
    aoi: ee.Geometry, start_year: int, end_year: int
) -> tuple[list[int], list[float]]:
    years = list(range(start_year, end_year + 1))
    mean_ndvi_values = []
    for year in years:
        image = add_indices(
            get_one_year_composite(aoi, year, start_month=6, end_month=8)
        )
        mean_ndvi = (
            image.select("NDVI")
            .reduceRegion(
                reducer=ee.Reducer.mean(),
                geometry=aoi,
                scale=30,
                bestEffort=True,
                maxPixels=1e9,
            )
            .get("NDVI")
            .getInfo()
        )
        mean_ndvi_values.append(mean_ndvi)
    return years, mean_ndvi_values

In [ ]:
import matplotlib.pyplot as plt


def plot_ndvi_trend(
    years: list[int], mean_ndvi_values: list[float], city_name: str
) -> None:
    plt.plot(years, mean_ndvi_values, marker="o")
    plt.xlabel("Year")
    plt.ylabel("Mean NDVI")
    plt.title(f"Mean NDVI trend: {city_name}")
    plt.grid(True)
    plt.show()


years, mean_ndvi_values = compute_ndvi_trend(aoi, START_YEAR, END_YEAR)
plot_ndvi_trend(years, mean_ndvi_values, CITY.name)